# M3 기존 양성 후보의 CLV shuffle 귀속 검정 - Dunnhumby

이 노트북은 기존 양성 후보 `m3_clv_allocated_relation_gate`의 개선이 **올바른 고객에게 배정된 historical CLV proxy**에서 나온 것인지 확인합니다. 같은 CLV 값들을 이진 사용자 차수 10분위 안에서 고객 사이에 재배정하고, 상품 기준값과 관계값·CLV 게이트를 모두 다시 계산합니다.

기존 양성 후보와 동일한 seed 42, validation 구간, 전파 변화 강도 0.075를 사용합니다. Test와 holdout은 만들지 않으며, 단일 시드의 사후 귀속 검정이므로 유의성이나 일반화를 주장하지 않습니다. 주 판정은 6개 정확도 지표(Recall/NDCG @10·20·50)의 기하평균 균형에서 실제 CLV가 M1과 CLV-shuffle을 모두 넘는지입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = 'cfdb3dab0004bb4287e49b823be2c7ef166d9f64'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_clv_relation import (
    configure_m3_clv_relation_shuffle_dunnhumby_run,
    shuffle_preflight_summary,
    run_shuffle_attribution_experiment,
)

cfg = configure_m3_clv_relation_shuffle_dunnhumby_run()
assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.'
summary = shuffle_preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['fixed']['target_propagation_strength'] == 0.075
assert summary['shuffle']['within'] == 'binary user-degree deciles'
assert summary['shuffle']['recomputes_item_baseline'] is True
assert summary['shuffle']['uses_shuffled_clv_in_relation_and_gate'] is True
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_shuffle_attribution_experiment(cfg)

In [ ]:
from IPython.display import display

display_df = result_df[result_df['split'].eq('val')].copy()
display_df = display_df.rename(columns={
    'revenue@10': 'price_purchase_amount_weighted_hit@10',
    'revenue@20': 'price_purchase_amount_weighted_hit@20',
    'revenue@50': 'price_purchase_amount_weighted_hit@50',
    'arp@10': 'mean_recommended_price_percentile@10',
})
columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'mean_recommended_price_percentile@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
]
available = [column for column in columns if column in display_df.columns]
display(display_df[available])
print('\nCLV 귀속 판정:')
print(json.dumps(result_df.attrs['attribution_decision'], ensure_ascii=False, indent=2))
print('\n기존 양성 후보 출처:', result_df.attrs['source_origin'])
print('결과 파일:', result_df.attrs['result_paths'])